# SPARK 2026 Mini Challenge
## Team Zambia - Chest X-Ray Pneumonia Classification

### Task
Build a CNN from scratch in PyTorch to classify chest X-rays as:
- **0** = Normal
- **1** = Pneumonia

### Evaluation Metric
Macro-averaged F1-score

### Dataset Citation
Kermany, D., Goldbaum, M., Cai, W. et al. (2018).
Identifying Medical Diagnoses and Treatable Diseases by Image-Based Deep Learning.
Cell, 172(5), 1122-1131.
https://doi.org/10.1016/j.cell.2018.02.010

In [10]:
# SPARK 2026 Mini Challenge
# Team Zambia - Chest X-Ray Pneumonia Classification
#
# Task:
# Build a CNN from scratch in PyTorch to classify chest X-rays as:
# 0 = Normal
# 1 = Pneumonia
#
# Evaluation metric:
# Macro-averaged F1-score
#
# Dataset citation:
# Kermany, D., Goldbaum, M., Cai, W. et al. (2018).
# Identifying Medical Diagnoses and Treatable Diseases by Image-Based Deep Learning.
# Cell, 172(5), 1122-1131.
# https://doi.org/10.1016/j.cell.2018.02.010

## Cell 2 — Imports

In [11]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from sklearn.metrics import f1_score, confusion_matrix, classification_report

## Cell 3 — Reproducibility

In [12]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# CRITICAL: GPU is essential for 35+ epochs to reach 98% accuracy
if device.type == "cpu":
    print("\n⚠️  WARNING: Training on CPU. To reach 98% accuracy in reasonable time (~2-3 hours),")
    print("   switch to GPU (Kaggle Notebook, Google Colab, or local CUDA setup).")
    print("   Current CPU-only training will take 8+ hours for 35 epochs.\n")

Using device: cpu

⚠️  WARNING: Training on CPU. To reach 98% accuracy in reasonable time (~2-3 hours),
   switch to GPU (Kaggle Notebook, Google Colab, or local CUDA setup).
   Current CPU-only training will take 8+ hours for 35 epochs.



## Cell 4 — Load CSV files

In [13]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(val_df.head())
display(test_df.head())

Train shape: (4099, 4)
Val shape: (878, 4)
Test shape: (879, 3)


,id,filepath,filename,target
0,CXR_05624,train/PNEUMONIA/CXR_05624.jpeg,CXR_05624.jpeg,1
1,CXR_01796,train/PNEUMONIA/CXR_01796.jpeg,CXR_01796.jpeg,1
2,CXR_04612,train/PNEUMONIA/CXR_04612.jpeg,CXR_04612.jpeg,1
3,CXR_02637,train/PNEUMONIA/CXR_02637.jpeg,CXR_02637.jpeg,1
4,CXR_02885,train/NORMAL/CXR_02885.jpeg,CXR_02885.jpeg,0


,id,filepath,filename,target
0,CXR_00918,val/NORMAL/CXR_00918.jpeg,CXR_00918.jpeg,0
1,CXR_04666,val/PNEUMONIA/CXR_04666.jpeg,CXR_04666.jpeg,1
2,CXR_01884,val/PNEUMONIA/CXR_01884.jpeg,CXR_01884.jpeg,1
3,CXR_00359,val/PNEUMONIA/CXR_00359.jpeg,CXR_00359.jpeg,1
4,CXR_01545,val/PNEUMONIA/CXR_01545.jpeg,CXR_01545.jpeg,1


,id,filepath,filename
0,CXR_05741,test/CXR_05741.jpeg,CXR_05741.jpeg
1,CXR_03646,test/CXR_03646.jpeg,CXR_03646.jpeg
2,CXR_00870,test/CXR_00870.jpeg,CXR_00870.jpeg
3,CXR_04969,test/CXR_04969.jpeg,CXR_04969.jpeg
4,CXR_01893,test/CXR_01893.jpeg,CXR_01893.jpeg


## Cell 5 — Basic integrity checks

In [14]:
print("Train class counts:")
print(train_df["target"].value_counts().sort_index())

print("\nVal class counts:")
print(val_df["target"].value_counts().sort_index())

print("\nDuplicate IDs in train:", train_df["id"].duplicated().sum())
print("Duplicate IDs in val:", val_df["id"].duplicated().sum())
print("Duplicate IDs in test:", test_df["id"].duplicated().sum())

Train class counts:
target
0    1108
1    2991
Name: count, dtype: int64

Val class counts:
target
0    237
1    641
Name: count, dtype: int64

Duplicate IDs in train: 0
Duplicate IDs in val: 0
Duplicate IDs in test: 0


## Cell 6 — Define transforms

In [15]:
train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
])

val_test_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
])

## Cell 7 — Custom dataset

In [16]:
class ChestXRayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "filepath"]
        image = Image.open(img_path).convert("L")  # grayscale
        # Convert grayscale to RGB (3-channel) for ImageNet normalization
        image_rgb = Image.new("RGB", image.size)
        image_rgb.paste(image)

        if self.transform:
            image_rgb = self.transform(image_rgb)

        if "target" in self.df.columns:
            label = torch.tensor(int(self.df.loc[idx, "target"]), dtype=torch.long)  # Class index for CrossEntropyLoss
            return image_rgb, label
        else:
            image_id = self.df.loc[idx, "id"]
            return image_rgb, image_id

## Cell 8 — Weighted sampler for class imbalance

In [17]:
class_counts = train_df["target"].value_counts().sort_index().values
total_samples = len(train_df)
num_classes = 2

print("Class counts [Normal, Pneumonia]:", class_counts)

# Correct formula: scale weights proportionally to class representation
class_weights = total_samples / (num_classes * class_counts)
print("Class weights:", class_weights)

# Store as tensor for use in CrossEntropyLoss (Gold Standard: use class weights in loss)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

sample_weights = train_df["target"].map({
    0: class_weights[0],
    1: class_weights[1]
}).values

# Copy the array to avoid PyTorch non-writable tensor warning
sample_weights = torch.DoubleTensor(sample_weights.copy())

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

Class counts [Normal, Pneumonia]: [1108 2991]
Class weights: [1.84972924 0.68522233]


## Cell 9 — DataLoaders

In [18]:
use_cuda = torch.cuda.is_available()
BATCH_SIZE = 32

train_dataset = ChestXRayDataset(train_df, transform=train_transforms)
val_dataset = ChestXRayDataset(val_df, transform=val_test_transforms)
test_dataset = ChestXRayDataset(test_df, transform=val_test_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=0,
    pin_memory=use_cuda
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=use_cuda
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=use_cuda
)

print(f"Using CUDA: {use_cuda}")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Using CUDA: False
Train batches: 129
Val batches: 28
Test batches: 28


## Cell 10 — CNN architecture

In [ ]:
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Gold Standard: 2 MaxPool to maintain 32x32 feature map (128 -> 64 -> 32)
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 128 -> 64
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64 -> 32
            
            # Block 3 (no pooling - preserve 32x32)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        # Gold Standard: 2 output nodes for binary classification with CrossEntropyLoss
        # OPTIMIZED: Use AdaptiveAvgPool2d(4) to balance spatial detail with computational efficiency
        # 32x32 -> 4x4 keeps local texture patterns (pneumonia opacities) while reducing compute 64x
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(4),  # 32x32 -> 4x4 = 2,048 features (vs 131K for full flatten)
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(2048, 256),  # Manageable layer size for fast training
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 2)  # 2 output nodes for Normal and Pneumonia
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = PneumoniaCNN().to(device)
print(model)

PneumoniaCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(64, 

## Cell 11 — Loss, optimizer, scheduler

In [20]:
# Gold Standard: Use class weights in loss function to handle imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.1,
    patience=7  # Gold Standard: Increase patience to allow convergence
)

## Cell 12 — Training function

In [21]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)  # CrossEntropyLoss expects class indices

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(loader)
    return epoch_loss

## Cell 13 — Evaluation function

In [22]:
def evaluate(model, loader, device, threshold=0.5):
    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)  # 2-class output

            # Get softmax probabilities for class 1 (Pneumonia)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy().ravel()
            preds = (probs >= threshold).astype(int)
            true = labels.cpu().numpy().astype(int).ravel()

            y_prob.extend(probs.tolist())
            y_pred.extend(preds.tolist())
            y_true.extend(true.tolist())

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return macro_f1, y_true, y_pred, y_prob

## Cell 14 — Threshold tuning

In [23]:
def find_best_threshold(model, loader, device):
    model.eval()

    y_true = []
    y_prob = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)  # 2-class output

            # Get softmax probabilities for class 1 (Pneumonia)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy().ravel()
            true = labels.cpu().numpy().astype(int).ravel()

            y_prob.extend(probs.tolist())
            y_true.extend(true.tolist())

    best_threshold = 0.5
    best_f1 = 0.0

    for threshold in np.arange(0.10, 0.91, 0.05):
        preds = (np.array(y_prob) >= threshold).astype(int)
        score = f1_score(y_true, preds, average="macro")

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    return best_threshold, best_f1

## Cell 15 — Training loop with best model saving

In [ ]:
EPOCHS = 35  # Gold Standard: Increase to 35 epochs for convergence
best_val_f1 = 0.0
best_epoch = 0

history = {
    "train_loss": [],
    "val_f1": []
}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_f1, _, _, _ = evaluate(model, val_loader, device, threshold=0.5)

    scheduler.step(val_f1)

    history["train_loss"].append(train_loss)
    history["val_f1"].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        torch.save(model.state_dict(), "best_gold_standard.pt")

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation Macro-F1 @0.5 threshold: {best_val_f1:.4f}")

Epoch 01 | Train Loss: 1.2608 | Val Macro-F1: 0.8586
Epoch 02 | Train Loss: 0.1831 | Val Macro-F1: 0.8984
Epoch 03 | Train Loss: 0.1633 | Val Macro-F1: 0.8562


## Cell 16 — Load best model

In [ ]:
model.load_state_dict(torch.load("best_gold_standard.pt", map_location=device))
model.eval()
print("Best Gold Standard model loaded.")

## Cell 17 — Tune threshold on validation set

In [ ]:
best_threshold, tuned_f1 = find_best_threshold(model, val_loader, device)

print("Best threshold:", best_threshold)
print("Best validation Macro-F1 after threshold tuning:", tuned_f1)

## Cell 18 — Final validation metrics

In [ ]:
from sklearn.metrics import recall_score

final_f1, y_true, y_pred, y_prob = evaluate(
    model,
    val_loader,
    device,
    threshold=best_threshold
)

pneumonia_recall = recall_score(y_true, y_pred, pos_label=1)
normal_recall = recall_score(y_true, y_pred, pos_label=0)

print("Final Validation Macro-F1:", final_f1)
print(f"Pneumonia Recall: {pneumonia_recall:.4f}")
print(f"Normal Recall:    {normal_recall:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

## Cell 19 — Generate test predictions

In [ ]:
model.eval()
results = []

with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)
        outputs = model(images)

        # Get softmax probabilities for class 1 (Pneumonia)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy().ravel()
        preds = (probs >= best_threshold).astype(int)

        for image_id, pred in zip(image_ids, preds):
            results.append({
                "id": image_id,
                "target": int(pred)
            })

submission = pd.DataFrame(results)
submission.head()

## Cell 20 — Validate submission format

In [ ]:
print("Submission shape:", submission.shape)
print("Expected shape:", sample_sub.shape)

print("\nSubmission columns:", submission.columns.tolist())
print("Expected columns:", sample_sub.columns.tolist())

print("\nMissing IDs:", set(sample_sub["id"]) - set(submission["id"]))
print("Extra IDs:", set(submission["id"]) - set(sample_sub["id"]))

## Cell 21 — Save submission file

In [ ]:
submission = submission[["id", "target"]]
submission.to_csv("Team_MedAi_Zambia_submission1.csv", index=False)

print("Submission file saved as Team_MedAi_Zambia_submission1.csv")

## Cell 22 — Optional notes for summary PDF

In [ ]:
# Run threshold tuning FIRST
best_threshold, tuned_f1 = find_best_threshold(model, val_loader, device)

# Build dynamic summary
summary_notes = f"""

GOLD STANDARD ARCHITECTURE — Targeting 98% Macro-F1 Score

Architecture:
- Custom CNN built from scratch using PyTorch (torch.nn)
- 3 convolutional blocks with 2 MaxPool layers (Gold Standard pattern)
- Spatial dimensions: 128 → 64 → 32 (preserves fine pneumonia detail)
- Input: 3-channel RGB (converted from grayscale for ImageNet normalization)
- Output: 2 nodes (Standard, Pneumonia) + CrossEntropyLoss
- Dropout: 0.4 and 0.3 for regularization

Preprocessing (Medical Standard):
- Images loaded via CSV metadata (filepath-based)
- Converted from grayscale to 3-channel RGB
- Resized to 128x128 pixels
- Normalized with ImageNet statistics: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
- Data augmentation: horizontal flip and 7° rotation on training set

Imbalance handling:
- WeightedRandomSampler used to address class imbalance

Training (Convergence Strategy):
- Loss function: CrossEntropyLoss (Gold Standard)
- Optimizer: Adam (lr=1e-3)
- Learning rate scheduling: ReduceLROnPlateau (patience=7)
- Total epochs: 35 (doubled from original 15 for near-zero loss convergence)
- Best model checkpoint saved as 'best_gold_standard.pt'

Evaluation:
- Primary metric: macro-averaged F1-score
- Threshold tuning on validation set (range: 0.10–0.90)

Expected Performance:
- Target: 98% accuracy / high Macro-F1
- Approach: Preserve spatial detail (32x32 maps) + drive loss toward zero + extended convergence time

Current Best Validation Performance:
- Best epoch: {best_epoch}
- Macro-F1 (@0.5 threshold): {best_val_f1:.4f}
- Optimal threshold: {best_threshold:.2f}
- Final Macro-F1 after threshold tuning: {tuned_f1:.4f}

"""

print(summary_notes)